# 🎙️ SLAM-ASR Turkish: Common Voice 27.0 Benchmark Notebook

Bu notebook, **SLAM-LLM** (Speech-Language Model Framework) metodolojisini birebir uygulayarak **Mozilla Common Voice 27.0 Türkçe** veri setinde state-of-the-art (SOTA) ASR başarımını elde etmeniz için tasarlanmıştır.

### ⚡ Özellikler:
- Kodlar doğrudan **GitHub** üzerinden Colab ortamına indirilir (Google Drive gerektirmez).
- Veri seti doğrudan **Mozilla Data Collective (MDC)** API üzerinden `CV_API` anahtarı ile otomatik indirilir.
- Eğitilen model ağırlıkları ve checkpoint'ler doğrudan **Hugging Face Hub** hesabınıza otomatik yüklenir.
- **Speech Encoder:** `openai/whisper-large-v3` (Frozen)
- **Projector:** 5x Frame Concatenation Linear Projector (Trainable)
- **LLM Backbone:** `Qwen/Qwen2.5-7B-Instruct` (LoRA Fine-tuning)

---

## 1. Donanım & Ekran Kartı Kontrolü (GPU Check)

In [1]:
!nvidia-smi

Wed Oct  7 19:09:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Projenin GitHub'dan Colab Ortamına İndirilmesi

In [2]:
%cd /content
!git clone https://github.com/gokhanersoy/slam-asr.git || true
%cd /content/slam-asr/commonvoice_tr

/content
Cloning into 'slam-asr'...
remote: Enumerating objects: 693, done.
remote: Counting objects: 100% (693/693), done.
remote: Compressing objects: 100% (505/505), done.
remote: Total 693 (delta 162), reused 674 (delta 143), pack-reused 0 (from 0)
Receiving objects: 100% (693/693), 30.71 MiB | 28.23 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/slam-asr/commonvoice_tr


## 3. Gerekli Kütüphanelerin Kurulumu

In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 141.0 MB/s eta 0:00:00


## 4. API Anahtarları ve Yetkilendirme 🔑

- Colab Secrets (Gizli Anahtarlar) alanına eklediğiniz `CV_API` (Mozilla Data Collective) ve `HF_TOKEN` (Hugging Face) otomatik yüklenir.

In [4]:
import os
try:
    from google.colab import userdata
    cv_api = userdata.get('CV_API')
    if cv_api:
        os.environ['CV_API'] = cv_api
        print("✅ CV_API Mozilla Data Collective anahtarı yüklendi!")

    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ HF_TOKEN Hugging Face anahtarı yüklendi!")
except Exception as e:
    print(f"Bilgi: {e}")

✅ CV_API Mozilla Data Collective anahtarı yüklendi!


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF_TOKEN Hugging Face anahtarı yüklendi!


## 5. Türkçe Common Voice 27.0 Veri Setinin İndirilmesi & Hazırlanması
Mozilla Data Collective API üzerinden `CV_API` ile Common Voice 27.0 Türkçe arşivi indirilir, ayıklanır, 16kHz WAV formatına dönüştürülür ve JSONL manifests dosyaları (`train.jsonl`, `val.jsonl`, `test.jsonl`) üretilir.

In [5]:
# Hızlı deneme yapmak isterseniz --max_train_samples 10000 ekleyebilirsiniz.
# Tüm veri seti ile eğitmek için parametresiz çalıştırın:
!python prepare_data.py --output_dir data

Fetching presigned download URL from Mozilla Data Collective...
Downloading: 100% 2.79G/2.79G [01:12<00:00, 41.4MiB/s]
✅ Download completed!
Extracting archive data/cv27_tr.tar.gz -> data/raw_extracted ...
/content/slam-asr/commonvoice_tr/prepare_data.py:75: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)
✅ Extraction completed!
Processing dataset from data/raw_extracted/cv-corpus-27.0-2026-09-11/tr ...
Processing train: 100% 41343/41343 [04:06<00:00, 167.74it/s]
[TRAIN] Exported 41343 items (45.14 hours) -> data/train.jsonl
Processing val: 100% 11591/11591 [01:07<00:00, 171.42it/s]
[VAL] Exported 11591 items (12.07 hours) -> data/val.jsonl
Processing test: 100% 11874/11874 [01:15<00:00, 157.06it/s]
[TEST] Exported 11874 items (14.54 hours) -> data/test.jsonl

🎉 Common Voice 27.0 Turkish Dataset Preparation Complete!


## 6. Model Eğitimi (SLAM-ASR Fine-Tuning)
Whisper Large V3 dondurulur, 2 katmanlı Linear Projector katmanı eğitilir ve Qwen2.5-7B-Instruct modeline LoRA uygulanır.

In [6]:
!python train.py train_config.num_epochs=5 train_config.batch_size_training=4 train_config.gradient_accumulation_steps=4

SLAM-ASR Training Start (Turkish Common Voice)
model_config:
  file: src/models/slam_model.py:model_factory
  llm_name: Qwen/Qwen2.5-7B-Instruct
  llm_path: Qwen/Qwen2.5-7B-Instruct
  llm_dim: 3584
  encoder_name: whisper
  encoder_path: openai/whisper-large-v3
  encoder_dim: 1280
  encoder_projector: linear
  encoder_projector_ds_rate: 5
  whisper_decode: false
  encoder_path_hf: null
dataset_config:
  dataset: speech_dataset
  train_data_path: data/train.jsonl
  val_data_path: data/val.jsonl
  test_data_path: data/test.jsonl
  input_type: mel
  mel_size: 128
  prompt: Aşağıdaki konuşma sesini Türkçe metne dönüştürün. Sadece konuşmanın tam
    transcriptini yazın.
  fix_length_audio: -1
  inference_mode: false
train_config:
  model_name: slam_asr_tr
  output_dir: checkpoints/slam_qwen_tr
  batch_size_training: 4
  val_batch_size: 4
  gradient_accumulation_steps: 4
  num_epochs: 5
  warmup_steps: 500
  total_steps: 10000
  lr: 0.0001
  validation_interval: 500
  freeze_encoder: true
  

## 7. Test Setinde Değerlendirme (WER & CER Hesaplama)

In [7]:
!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --output_file evaluation_results.json

Evaluating SLAM-ASR Turkish on device: cuda
Loading weights: 100% 339/339 [00:01<00:00, 274.52it/s]
Running batch decoding...
  0% 0/2969 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(
  0% 0/2969 [00:02<?, ?it/s]
Traceback (most recent call last):
  File "/content/slam-asr/commonvoice_tr/evaluate.py", line 119, in <module>
    main()
    ~~~~^^
  File "/content/slam-asr/commonvoice_tr/evaluate.py", line 70, in main
    pred_text = model.generate_transcription(
        audio_mel=mel,
        prompt=prompt_str,
        max_new_tokens=128
    )
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
  File "/content/slam-asr/commonvoice_tr/src/models/slam_model.py", lin

## 8. Eğitilen Modeli Hugging Face Hub'a Yükleme 🚀
Eğitilen model ağırlıklarını tek komutla kendi Hugging Face hesabınıza yükleyin.

In [8]:
# Kendi Hugging Face Kullanıcı Adınızı Yazın (Örn: gokhanersoy/slam-asr-tr-commonvoice)
HF_REPO_ID = "gokhanersoy/slam-asr-tr-commonvoice"

!python push_to_hub.py --repo_id {HF_REPO_ID} --checkpoint_dir checkpoints/slam_qwen_tr

Creating/verifying repository: gokhanersoy/slam-asr-tr-commonvoice ...
Uploading files from checkpoints/slam_qwen_tr to Hugging Face Hub (gokhanersoy/slam-asr-tr-commonvoice) ...
Found 0 files to upload
  Preparing   ░░░░░░░░░░░░░░░░░░░░  0 / 0 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 0
No files have been modified since last commit. Skipping to prevent empty commit.
✅ Upload Complete! Model available at: https://huggingface.co/gokhanersoy/slam-asr-tr-commonvoice


## 9. Örnek Deşifre Playground (Single Audio Inference)

In [9]:
import torch
import whisper
from omegaconf import OmegaConf
from src.models.slam_model import model_factory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = OmegaConf.load("conf/prompt.yaml")

# Modeli yükle
model, tokenizer = model_factory(
    train_config=cfg.train_config,
    model_config=cfg.model_config,
    ckpt_path="checkpoints/slam_qwen_tr/best_checkpoint.pt"
)
model.to(device)
model.eval()

# Örnek Ses Dosyası Yolu
sample_audio_path = "data/audio/test/test_000000.wav"

# Log Mel Spectrogram
audio_raw = whisper.load_audio(sample_audio_path)
audio_raw = whisper.pad_or_trim(audio_raw)
audio_mel = whisper.log_mel_spectrogram(audio_raw, n_mels=128).permute(1, 0).unsqueeze(0).to(device)

# Tahmin Al
prediction = model.generate_transcription(audio_mel=audio_mel)
print(f"🎙️ Tahmin Edilen Metin: {prediction}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1455: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(


RuntimeError: expected scalar type Float but found BFloat16